<a href="https://colab.research.google.com/github/tmoura/EDES/blob/main/Percurso_Grafos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
class Aresta():

  def __init__(self, origem, destino, peso):
    self.__origem = origem
    self.__destino = destino
    self.__peso = peso

  @property
  def origem(self):
    return self.__origem


  @origem.setter
  def origem(self, novaOrigem):
    self.__origem = novaOrigem

  @property
  def destino(self):
    return self.__destino


  @destino.setter
  def destino(self, novoDestino):
    self.__destino = novoDestino

  @property
  def peso(self):
    return self.__peso

  @peso.setter
  def peso(self, peso: int):
    self.__peso = peso

  def __str__(self):
    return f'{self.__origem},{self.__destino},{self.__peso}'


class Vertice():

  def __init__(self, dado):
    self.__dado = dado
    self.__adj = []

  def addAdj(self, edge):
    self.__adj.append(edge)

  @property
  def dado(self):
    return self.__dado

  @dado.setter
  def dado(self, dado):
    self.__dado = dado

  @property
  def adj(self):
    return self.__adj

  def __str__(self):
    return self.__dado


In [43]:
class Grafo():

	def __init__(self):
		self.__vertices = []
		self.__arestas = []

	def addVertice(self, nome):
		v = Vertice(nome)
		self.__vertices.append(v)
		return v

	def addAresta(self, origem, destino, peso):
		e = Aresta(origem, destino, peso)
		origem.addAdj(e)
		self.__arestas.append(e)
		return e

	def __str__(self):
		r = ''
		for u in self.__vertices:
			if len(u.adj) == 0:
				continue
			r += (str(u.dado) + ' -> ')
			for e  in u.adj:
				v = e.destino
				r += v.dado + ', '
			r += '\n'
		return r

	def getNumVertices(self):
		return len(self.__vertices)

	def getNumArestas(self):
		return len(self.__arestas)

	def breadth_first_search(self, inicio):
		visitado = []
		fila = []
		saida = []

		fila.append(inicio)
		visitado.append(inicio)

		while fila != []:
			atual = fila.pop(0) # processa o vértice
			saida.append(atual.dado)

			for edge in atual.adj:
				neighbor = edge.destino

				if neighbor not in visitado:
					visitado.append(neighbor)
					fila.append(neighbor)
		return saida

	def depth_first_search(self, inicio):
		visitado = []
		pilha = []
		saida = []

		pilha.append(inicio)

		while pilha != []:
			atual = pilha.pop(0)

			if atual not in visitado:
				visitado.append(atual)
				saida.append(atual.dado)

				for edge in reversed(atual.adj):
					neighbor = edge.destino
					if neighbor not in visitado:
						pilha.append(neighbor)
		return saida

	def depth_first_search_recursivo(self, vertice, visitado, saida):
		visitado.append(vertice)
		saida.append(vertice.dado)

		for edge in vertice.adj:
			neighbor = edge.destino
			if neighbor not in visitado:
				self.depth_first_search_recursivo(neighbor,visitado, saida)

		return saida

	def dijkstra(self, inicio):
		distancias = {v: float('infinity') for v in self.__vertices}

		#getNumVertices(self)
		distancias[inicio] = 0

		unvisited_vertices = list(self.__vertices) # List of all vertices initially unvisited

		caminhos = {v: None for v in self.__vertices}

		while unvisited_vertices:
			vertice_atual = None
			min_distancia = float('infinity')

			# Find the unvisited vertex with the smallest distance
			for v in unvisited_vertices:
				if distancias[v] < min_distancia:
					min_distancia = distancias[v]
					vertice_atual = v

			# If no reachable vertex is found, break
			if vertice_atual is None:
				break

			unvisited_vertices.remove(vertice_atual)

			for aresta in vertice_atual.adj:
				vizinho = aresta.destino
				peso = aresta.peso

				if vizinho in unvisited_vertices: # Only consider unvisited neighbors
					distancia = distancias[vertice_atual] + peso

					if distancia < distancias[vizinho]:
						distancias[vizinho] = distancia
						caminhos[vizinho] = vertice_atual

		return distancias, caminhos

	def dijkstra_novo(self, origem, destino):
			vertices = self.__vertices  # acessando lista interna
			n = len(vertices)

			dist = [float('inf')] * n
			visitado = [False] * n
			anterior = [None] * n

			origem_idx = vertices.index(origem)
			destino_idx = vertices.index(destino)

			dist[origem_idx] = 0

			for _ in range(n):
					# 1️⃣ Escolher o vértice não visitado com menor distância
					menor_dist = float('inf')
					u_idx = -1

					for i in range(n):
							if not visitado[i] and dist[i] < menor_dist:
									menor_dist = dist[i]
									u_idx = i

					if u_idx == -1:
							break

					visitado[u_idx] = True
					u = vertices[u_idx]

					# 2️⃣ Relaxar as arestas
					for aresta in u.adj:
							v = aresta.destino
							v_idx = vertices.index(v)

							if not visitado[v_idx]:
									nova_dist = dist[u_idx] + aresta.peso
									if nova_dist < dist[v_idx]:
											dist[v_idx] = nova_dist
											anterior[v_idx] = u

			# 🔹 Reconstrução do caminho
			caminho = []
			atual = destino

			while atual is not None:
					caminho.append(atual)
					atual = anterior[vertices.index(atual)]

			caminho.reverse()

			if caminho[0] != origem:
					return None, float('inf')

			return caminho, dist[destino_idx]

	def dijkstra_thiago(self, origem, destino):
			vertices = self.__vertices  # lista interna dos vértices
			n = len(vertices)

			dist = [float('inf')] * n
			visitado = [False] * n
			anterior = [None] * n

			dist[vertices.index(origem)] = 0

			for j in range(n):
					# 1️) Escolher o vértice não visitado com menor distância. Será escolhido primeiro a origem
					menor_dist = float('inf')
					index = -1

					for i in range(n):
							if not visitado[i] and dist[i] < menor_dist:
									menor_dist = dist[i]
									index = i

					if index == -1:
							break

					visitado[index] = True
					u = vertices[index]

					# 2️) Tenta melhorar o caminho para os vizinhos. Escolhendo o mais curto
					for aresta in u.adj:
							v = aresta.destino
							v_idx = vertices.index(v)

							if not visitado[v_idx]:
									nova_dist = dist[index] + aresta.peso
									if nova_dist < dist[v_idx]:
											dist[v_idx] = nova_dist
											anterior[v_idx] = u

			# 3) Reconstrução do caminho
			caminho = []
			atual = destino

			while atual is not None:
					caminho.append(atual)
					atual = anterior[vertices.index(atual)]

			caminho.reverse()

			if caminho[0] != origem:
					return None, float('inf')

			return caminho, dist[vertices.index(destino)]

In [16]:
g = Grafo()

um = g.addVertice('1')
dois = g.addVertice('2')
tres = g.addVertice('3')
quatro = g.addVertice('4')
cinco = g.addVertice('5')
seis = g.addVertice('6')

g.addAresta(seis,quatro,1)
g.addAresta(quatro,cinco,1)
g.addAresta(quatro,tres,1)
g.addAresta(tres,dois,1)
g.addAresta(cinco,dois,1)
g.addAresta(dois,um,1)

print(f"BFS ordem (iterativo): {g.breadth_first_search(seis)}")

print(f"DFS ordem (iterativo): {g.depth_first_search(seis)}")
print(f"DFS ordem (recursivo): {g.depth_first_search_recursivo(seis,[],[])}")

BFS ordem (iterativo): ['6', '4', '5', '3', '2', '1']
DFS ordem (iterativo): ['6', '4', '3', '5', '2', '1']
DFS ordem (recursivo): ['6', '4', '5', '2', '1', '3']


In [44]:
g2 = Grafo()

a = g2.addVertice('A')
b = g2.addVertice('B')
c = g2.addVertice('C')
d = g2.addVertice('D')
e = g2.addVertice('E')

g2.addAresta(a,b,1)
g2.addAresta(a,d,4)
g2.addAresta(b,d,5)
g2.addAresta(b,c,2)
g2.addAresta(c,e,1)
g2.addAresta(d,e,1)

caminho, custo = g2.dijkstra_thiago(a, e)

print("Caminho:")
for v in caminho:
    print(v.dado, end=" ")

print("\nCusto:", custo)

Caminho:
A B C E 
Custo: 4
